# Vaani Track 1 - v2 on Kaggle

ATST-Frame + BEATs fusion -> trident span-regression head -> count-head selection ->
per-district calibration.

## Before you run anything

1. **Accelerator:** *Settings -> Accelerator -> GPU T4 x2*.
2. **Internet:** *Settings -> Internet -> On* (needed to clone and to fetch checkpoints).
3. **Secrets** (*Add-ons -> Secrets*):
   - `HF_TOKEN` - the dataset is gated; accept the terms on the dataset page first.
   - `GITHUB_TOKEN` - **not needed.** The repo is public. Only add one if you make it
     private again, and then set `REPO_IS_PRIVATE = True`.
4. **Everything you edit lives in the CONFIG cell below.** Nothing further down needs
   changing.

## About session limits

A Kaggle GPU session stops at ~12 h, and 60 epochs over 154 h of audio will not fit in
one. The run is built for that: it writes `state.pt` every epoch, and `--resume auto`
picks it back up with the optimiser, the EMA and the **step counter** intact. Without
the step counter the cosine schedule would restart and the learning rate would jump
back up, quietly undoing the previous session's progress.

The pattern across sessions is: *Save Version* -> next session, add this notebook's
output under *Add Data* -> point `RESUME_FROM` at it. Do the same for the prepared
corpus via `DATA_FROM`, so you download 16.5 GB once rather than every session.

In [ ]:
# ============================== CONFIG ==============================
# The only cell you need to edit.

REPO = "raut7218/vaani-sed-v2"
REPO_IS_PRIVATE = False       # repo is public; set True + add a GITHUB_TOKEN secret if that changes

# --- data ---------------------------------------------------------------
# First session: leave DATA_FROM empty to download the corpus.
# Later sessions: point it at a saved Dataset / notebook output to skip the download.
DATA_FROM  = ""               # e.g. "/kaggle/input/vaani-prepared"
MAX_SHARDS = 0                # 0 = all 182 shards (~16.5 GB). Try 4 for a first pass.

# --- training -----------------------------------------------------------
FOLD        = 0               # state-grouped k-fold; train several and ensemble
EPOCHS      = 60
BATCH_SIZE  = 16              # PER GPU, so 16 x 2 = 32 global under torchrun
RESUME_FROM = ""              # e.g. "/kaggle/input/vaani-run-f0" to continue a run

USE_VAD       = True          # speech-presence auxiliary head
USE_SYNTHETIC = True          # bronze tags -> strong labels by construction
N_SYNTHETIC   = 20000

# --- inference ----------------------------------------------------------
# Run the "find the evaluation audio" cell below, then paste the path here.
TEST_AUDIO_DIR = ""           # e.g. "/kaggle/input/indoml-track1-eval/audio"
# Checkpoints to ensemble. Leave empty to use whatever this session trained.
ENSEMBLE_CKPTS = []           # e.g. ["/kaggle/input/vaani-run-f1/best.pt"]
# =====================================================================

WORK  = "/kaggle/working"
DATA  = DATA_FROM or (WORK + "/data")
SYNTH = WORK + "/synth"
RUN   = WORK + "/runs/f%d" % FOLD
print("data:", DATA, "| run:", RUN)


## 1. Fetch the repo

The repo is public, so this is a plain clone - no token, no Kaggle secret.

The `REPO_IS_PRIVATE` branch is kept for the case where you make it private later: it reads a `GITHUB_TOKEN` secret and then strips the credentialed remote, so the token is not left sitting in `.git/config` inside a notebook output you might share.

In [ ]:
import os, subprocess, shutil
from pathlib import Path

SRC = WORK + "/v2"
shutil.rmtree(SRC, ignore_errors=True)

url = "https://github.com/%s.git" % REPO
if REPO_IS_PRIVATE:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://x-access-token:%s@github.com/%s.git" % (tok, REPO)

r = subprocess.run(["git", "clone", "--depth", "1", url, SRC],
                   capture_output=True, text=True)
# Never print the command or stderr verbatim - the token is in the URL.
if r.returncode != 0:
    raise SystemExit(
        "clone failed. Check: Internet is On, GITHUB_TOKEN exists and has read "
        "access to this repo, and REPO_IS_PRIVATE matches reality.")

# Drop the credentialed remote so the token does not persist in the saved output.
subprocess.run(["git", "-C", SRC, "remote", "set-url", "origin",
                "https://github.com/%s.git" % REPO], check=True)

os.chdir(SRC)
print("cloned to", SRC)


In [ ]:
!pip -q install -r requirements.txt 2>&1 | tail -2


## 2. Verify the wiring before spending GPU hours

`test_overfit.py` is the one that matters. It proves the span head is time-aligned to the audio and places boundaries to a few milliseconds. A silent offset between waveform and targets does not show up in the loss - the model simply learns the offset - and it costs the entire event-F1 term.

In [ ]:
!python tests/test_components.py | tail -3
!python tests/test_overfit.py | tail -4


## 3. Encoders

ATST-Frame (40 ms tokens) is the primary encoder; BEATs (160 ms) rides along as a semantic channel.

The loader **refuses to run** if under 90% of the ATST checkpoint's tensors land - a half-loaded encoder trains a partly-random network and still produces a perfectly plausible loss curve. If the weights cannot be fetched, training still runs BEATs-only, so read this cell's output before moving on.

In [ ]:
!python scripts/fetch_encoders.py --all
!ls -la checkpoints/ 2>/dev/null || echo "no checkpoints dir"


## 4. Data

Skipped entirely when `DATA_FROM` is set. Otherwise: 182 shards, ~16.5 GB parquet, 90,637 clips (~154.6 h), gated behind `HF_TOKEN`.

Set `MAX_SHARDS = 4` for a first end-to-end pass - it takes minutes instead of hours and confirms the whole pipeline runs on real audio before you commit a full session to it.

In [ ]:
if DATA_FROM:
    print("using prepared corpus at", DATA_FROM)
else:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    shards = "--max-shards %d" % MAX_SHARDS if MAX_SHARDS else ""
    get_ipython().system("python scripts/download_data.py --out %s %s" % (DATA, shards))

import json, collections
recs = [json.loads(l) for l in open(DATA + "/manifest.jsonl", encoding="utf-8") if l.strip()]
print(len(recs), "clips |", collections.Counter(r["tier"] for r in recs))
print("states:", len({r.get("state", "") for r in recs}))


## 5. Free supervision

**VAD** gives the speech head its pseudo-labels. Vaani is *speech* recordings; every DCASE-derived system treats it as a generic soundscape, and factoring speech out makes the noise boundaries much easier to place.

**Synthetic** turns the bronze tier's tag-only hours into strong labels: cut a segment from a clip tagged `animal_sound`, paste it at a known position, and the label is correct *by construction* - no confidence threshold, so no error to accumulate the way self-training does.

In [ ]:
if USE_VAD and not DATA_FROM:
    get_ipython().system("python scripts/make_vad.py --data %s" % DATA)
if USE_SYNTHETIC:
    get_ipython().system("python scripts/make_synthetic.py --data %s --out %s -n %d"
                         % (DATA, SYNTH, N_SYNTHETIC))


## 6. Build this session's config

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["data"]["vad_dir"]      = (DATA + "/vad") if USE_VAD else ""
cfg["model"]["beats_dir"]   = SRC + "/checkpoints"
cfg["train"]["num_workers"] = 2          # Kaggle gives 4 vCPUs alongside the 2 GPUs
pathlib.Path("configs/kaggle.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump({k: cfg[k] for k in ("data", "model")}, sort_keys=False))


## 7. Train

`--batch-size` is **per GPU** (the standard DDP convention), so 16 becomes 32 globally across the two T4s.

With `RESUME_FROM` set, the previous session's `state.pt` is copied in first and `--resume auto` continues from it: same optimiser state, same EMA, same step counter.

In [ ]:
extra = ("--extra-data " + SYNTH) if USE_SYNTHETIC else ""
resume = ""
if RESUME_FROM:
    Path(RUN).mkdir(parents=True, exist_ok=True)
    for f in ("state.pt", "best.pt", "history.json"):
        s = Path(RESUME_FROM) / f
        if s.exists():
            shutil.copy(s, Path(RUN) / f)
            print("restored", f)
    resume = "--resume auto"

get_ipython().system(
    "torchrun --standalone --nproc_per_node=2 -m src.train.train "
    "--config configs/kaggle.yaml --data %s %s --out %s --fold %d "
    "--epochs %d --batch-size %d %s"
    % (DATA, extra, RUN, FOLD, EPOCHS, BATCH_SIZE, resume))


### If the session is about to time out

Hit *Save Version* ("Save & Run All" or "Quick Save" - either preserves
`/kaggle/working`). Next session: *Add Data -> Your Notebooks -> this notebook's
output*, then set

```python
RESUME_FROM = "/kaggle/input/<this-notebook-output>/runs/f0"
DATA_FROM   = "/kaggle/input/<this-notebook-output>/data"
```

and run again. The `state.pt` written every epoch is what makes that lossless.

## 8. Diagnose

The headline score says almost nothing about *which* failure mode you are in. This prints the constant-baseline comparison (v1 lost to a one-line heuristic - always check), strict vs loose recall (found the event vs placed the boundaries), the boundary error distribution, the operating point against the corpus prior, and the selection oracle: how much is still available from better selection, and how much needs a better model.

In [ ]:
get_ipython().system("python scripts/diagnose.py --ckpt %s/best.pt --data %s --fold %d"
                     % (RUN, DATA, FOLD))


## 9. Find the evaluation audio

Run this, find your eval set in the listing, and paste the path into `TEST_AUDIO_DIR` in the CONFIG cell.

In [ ]:
for p in sorted(Path("/kaggle/input").glob("*")):
    audio = list(p.rglob("*.wav")) + list(p.rglob("*.flac"))
    print("%7d audio files   %s" % (len(audio), p))
    for sub in sorted({q.parent for q in audio[:400]}):
        print("                    ->", sub)


## 10. Submit

Several checkpoints are fused with **1D weighted box fusion**, not by averaging posteriors - averaging two models that localise an onset 80 ms apart widens the ramp by 80 ms, and fusing spans keeps the edges sharp.

Per-district calibration is transductive: it groups test clips by the `State_District` encoded in their filenames and matches each group's predicted event count and coverage to the corpus prior. It uses only unlabelled test audio, never a label.

**Watch the last line.** If `events/clip` and `coverage` are far from the priors (1.22 and 0.52), the operating point is wrong - and on this metric that is worth more than most model changes.

In [ ]:
assert TEST_AUDIO_DIR, "set TEST_AUDIO_DIR in the CONFIG cell (see the cell above)"
ckpts = " ".join(ENSEMBLE_CKPTS or [RUN + "/best.pt"])
get_ipython().system(
    "python -m src.infer.predict --ckpt %s --audio-dir %s --out %s/submission.zip "
    "--batch-size 16 --num-workers 2" % (ckpts, TEST_AUDIO_DIR, WORK))


In [ ]:
import zipfile, json, numpy as np
with zipfile.ZipFile(WORK + "/submission.zip") as z:
    assert z.namelist() == ["predictions.jsonl"], z.namelist()
    rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]

ids = [r["clip_id"] for r in rows]
assert len(ids) == len(set(ids)), "duplicate clip_id"
for r in rows:
    for e in r["events"]:
        assert e["onset"] >= 0 and e["offset"] >= e["onset"], r["clip_id"]

print("%d clips | %d events | %.2f events/clip | %.1f%% empty"
      % (len(rows), sum(len(r["events"]) for r in rows),
         np.mean([len(r["events"]) for r in rows]),
         100 * np.mean([not r["events"] for r in rows])))
print(rows[0])
print("\nDownload /kaggle/working/submission.zip from the Output tab -> upload to Codabench.")


## 11. Optional: more folds for the ensemble

Each fold holds out a different group of states. Train them in separate sessions, save
each output, then list all their `best.pt` paths in `ENSEMBLE_CKPTS`.

Selecting a model on one narrow state slice is what cost v1 0.16 between validation and
the leaderboard - folds are the fix, and they hand you the ensemble for free.

In [ ]:
# In a later session: set FOLD = 1 (then 2, ...) in CONFIG and re-run from the config cell.
# Then submit with, for example:
#   ENSEMBLE_CKPTS = ["/kaggle/input/vaani-f0/best.pt",
#                     "/kaggle/input/vaani-f1/best.pt",
#                     "/kaggle/input/vaani-f2/best.pt"]
